# Wrap-style hooks
### wrap_model_call
#### 基于装饰器实现
我们可以同时在模型调用前后做事，所以命名为 `wrap_model_call` ，wrap意为 包裹 。
```python
# 源码
def wrap_model_call(
request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """
    request: 包含 model, messages, system_message, tools, state
    handler: 执行实际模型调用的函数
    返回：ModelResponse
    """
```

In [2]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [3]:
# 基于装饰器实现
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable


@wrap_model_call
def wrap_model_call_middleware(
    request: ModelRequest,  # 包含即将发送给大模型的所有请求数据（如消息列表、温度等）
    handler: Callable[[ModelRequest], ModelResponse],  # 代表下一个中间件或最终的大模型调用
) -> ModelResponse | None:
    # 动态篡改用户发出的最后一条消息的内容，悄悄往里面追加字符串。
    # 典型应用：统一在底层为所有请求追加特殊的 Prompt 提示词（例如：“请用中文回答”、“禁止透漏公司机密”等）。
    request.messages[-1].content += " -> wrap_model_call_before <- "

    # 将修改后的请求传递给 handler，真正去调用大模型（或者流转到下一个拦截器）
    # 这一步会产生真实的 Token 消耗并等待大模型响应
    response = handler(request)

    # 大模型返回响应后，在将响应交付给 Agent 状态机之前，对其内容进行直接篡改
    # `response.result` 是一个消息列表，修改其第一条返回消息的内容
    # 典型应用：做底层的文本敏感词过滤、输出格式强行格式化、或是统一添加某些后处理标记。
    response.result[0].content += " -> wrap_model_call_after <- "

    # 将修改完的响应体返回，继续维持 Agent 生命周期流转
    return response


agent = create_agent(
    model=model,
    middleware=[wrap_model_call_middleware],
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> wrap_model_call_before <- 
================================== Ai Message ==================================

你好！👋

`-> wrap_model_call_before <-` 看起来像是你代码或框架中定义的**模型调用前置拦截钩子**（Hook / Wrapper）。这类命名在 LLM 工程化开发中很常见，通常用于在真实请求发往大模型之前插入自定义逻辑，例如：

- 🔍 **观测与调试**：记录入参、耗时、Token 消耗
- 🛠️ **动态改造输入**：注入 System Prompt、压缩历史、替换变量、敏感词过滤
- 📦 **缓存/降级**：命中缓存直接返回，或切换到备用模型
- 🔐 **鉴权/限流**：校验用户权限、配额控制、频率限制
- 🧩 **多路由决策**：按意图分发到不同能力或模态的模型

如果你是在某个具体框架（如 LangChain、LlamaIndex、自研 Agent 管道、vLLM/TGI 代理层等）里看到这个结构，或者想让我帮你写一个标准的 `wrap_model_call_before` 中间件/装饰器模板，可以告诉我：
1. 你使用的框架或运行时环境
2. 希望在这个阶段实现的具体功能
3. 输入输出的数据结构示例

随时配合你的架构风格来定制 💡  
（注：我本身是纯文本交互接口，没有内置调用链挂钩，但完全可以在应用层帮你封装这套逻辑。） -> wrap_model_call_after <-


In [4]:
# 基于类实现
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable


class WrapModelCallMiddleWare(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse | None:
        request.messages[-1].content += " -> wrap_model_call_before <- "
        response = handler(request)
        response.result[0].content += " -> wrap_model_call_after <- "
        return response


agent = create_agent(
    model=model,
    middleware=[WrapModelCallMiddleWare()],
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> wrap_model_call_before <- 
================================== Ai Message ==================================

你好！👋  
`-> wrap_model_call_before <-` 看起来像是在描述一个**模型调用前置钩子（Pre-call Hook）**。这类命名常见于自研推理服务、AI Agent 框架或中间件系统中，用于在真实请求到达底层大模型之前拦截并处理输入。

### 🔍 典型用途
| 功能 | 说明 |
|------|------|
| 📝 日志/埋点 | 记录请求时间、token 数、用户 ID 等 |
| 🛡️ 安全过滤 | 敏感词检测、注入攻击拦截、合规审查 |
| ⚡ 缓存加速 | 相同 prompt 直接返回历史结果 |
| 🔄 格式标准化 | 清洗多轮对话结构、补全系统提示词 |
| 📉 限流/降级 | QPS 控制、熔断触发、 fallback 路由 |

### 💡 基础实现示例（Python）
```python
def wrap_model_call_before(prompt: str, **kwargs) -> tuple[str, dict]:
    """前置包装器：在调用 llm.invoke() 前执行"""
    # 1. 输入清洗
    clean_prompt = prompt.strip().lower()
    
    # 2. 安全检查（示例）
    if any(keyword in clean_prompt for keyword in ["恶意", "泄露"]):
        raise ValueError("⚠️ 触发内容安全规则")
        
    # 3. 日志记录
    import time
    print(f"[BEFORE @ {time.time():.3f}] Prompt length: {len(clean_prompt)}")
    
    return 

使用场景：用于拦截、重试、缓存模型调用。

In [ ]:
# 失败重试
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
import time


@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """自动重试失败的模型调用"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            print(f"🔄 尝试调用模型（第 {attempt + 1}/{max_retries} 次）")
            return handler(request)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ 所有重试失败：{e}")
                raise
            # 指数退避
            wait_time = 2 ** attempt
            print(f"⚠ 调用失败：{e}，{wait_time} 秒后重试")
            time.sleep(wait_time)

In [ ]:
# 响应缓存
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.agents import create_agent
from typing import Callable
import hashlib
import json


class ModelCache:
    """模型响应缓存"""

    def __init__(self):
        self.cache = {}

    def create_hook(self):
        @wrap_model_call
        def cache_model(
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse:
            # 生成缓存键
            cache_key = hashlib.md5(
                json.dumps({
                    "messages": [str(m) for m in request.messages],
                    "system": str(request.system_message),
                }).encode()
            ).hexdigest()

            # 检查缓存
            if cache_key in self.cache:
                print("💾 缓存命中！")
                return self.cache[cache_key]

            # 调用模型
            print("🔍 缓存未命中，调用模型")
            response = handler(request)

            # 存入缓存
            self.cache[cache_key] = response
            return response

        return cache_model


# 使用
cache = ModelCache()
agent = create_agent(
    model=model,
    middleware=[cache.create_hook()],
)

In [ ]:
# 修改系统提示
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_core.messages import SystemMessage
from typing import Callable
from datetime import datetime


@wrap_model_call
def add_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """动态添加上下文信息到系统提示"""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # 构建新的系统消息
    original_content = request.system_message.content if request.system_message else ""
    new_content = f"""{original_content}
当前时间：{current_time}
用户位置：中国
语言偏好：中文
"""

    # 创建新的系统消息
    new_system_message = SystemMessage(content=new_content)

    # 使用 override 方法修改请求
    modified_request = request.override(system_message=new_system_message)
    return handler(modified_request)